In [ ]:
# ====================================================================================================================
# MT2KD — CELL 2 of 2: STAGE 2 (multi-task fine-tuning with labels) + FINAL EVALUATION on the official test sets
#   Run this AFTER Cell 1 has printed "CELL 1 FINISHED", in the SAME runtime.
#   After a runtime restart: run Cell 1 again first (Stage 1 is already finished, so it only rebuilds the local data/caches), then this cell.
#   Stage 2 (paper Sec. III-D): encoder frozen for the first updates (only the new heads learn), then everything is trained with the encoder
#   at 1/5 of the base learning rate.  ASR: CTC (characters)  |  AT: FSD50K 200 classes (BCE)  |  SV: speaker classification (CE)
#   After ANY interruption: run this cell again -> it resumes from the last Stage-2 checkpoint.
# ====================================================================================================================
_need = ["MA", "MB", "MC", "D", "DATA_SIG", "S1", "DRV1", "DRV2", "build_joint_model", "train_stage1"]
if any(n not in globals() for n in _need) or not (globals().get("S1") or {}).get("finished"):
    raise RuntimeError("Cell 1 has not finished in this runtime: run Cell 1 first (it skips everything already done), then this cell.")
# ####################################################################################################################
# ---- STAGE 2: multi-task supervised fine-tuning (labels): ASR = CTC (characters), AT = FSD50K 200 classes (BCE), SV = speaker classification (CE)
# ####################################################################################################################
def build_stage2_heads(H1, D, dev):
    """New task heads. AT: 512 -> 200 FSD50K classes, rows of the 192 mapped classes initialised from the Stage-1 AudioSet head (same AudioSet ID);
    SV: M_SV (from Stage 1) -> 192-D speaker embedding -> linear speaker classifier; ASR: each 25 Hz frame repeated twice -> linear -> 29 CTC symbols."""
    import torch, torch.nn as nn
    torch.manual_seed(E["seed"] + 2)
    H = nn.ModuleDict(dict(ctc=nn.Linear(512, len(VOCAB)), at=nn.Linear(512, 200), msv=H1["msv"], spk=nn.Linear(192, D["n_spk"]))).to(dev)
    with torch.no_grad():
        H["at"].weight[torch.as_tensor(D["F_IDX"])] = H1["at"].weight[torch.as_tensor(D["A_IDX"])].to(dev)
        H["at"].bias[torch.as_tensor(D["F_IDX"])] = H1["at"].bias[torch.as_tensor(D["A_IDX"])].to(dev)
    return H

def s2_batch_loss(task, b, student, H, D, dev, amp, frozen):
    """frozen=True: the encoder runs without gradient (head warm-up phase, paper: encoder frozen for the first updates)."""
    import torch, torch.nn.functional as F
    ctx = torch.no_grad() if frozen else torch.enable_grad()
    fb, fl = b["fb"].to(dev, non_blocking=True), b["fb_lens"].to(dev)
    if task == "sv":
        with ctx, torch.autocast("cuda", dtype=amp): y, yl = student.sv_features(fb, fl)
        emb = H["msv"](y.detach() if frozen else y, yl); lab = torch.as_tensor(D["spk_of"][b["i"].numpy()], device=dev)
        return F.cross_entropy(H["spk"](emb).float(), lab), {}
    with ctx, torch.autocast("cuda", dtype=amp): enc, el = student(fb, fl)
    if frozen: enc = enc.detach()
    if task == "at":
        with torch.autocast("cuda", dtype=amp): z = H["at"](masked_mean(enc, el))
        return F.binary_cross_entropy_with_logits(z.float(), b["t"].to(dev).float()), {}
    with torch.autocast("cuda", dtype=amp): lg = H["ctc"](enc.repeat_interleave(2, dim=1))
    lp = lg.float().log_softmax(-1); L = el * 2; tl = b["id_lens"].to(dev)
    loss = F.ctc_loss(lp.transpose(0, 1), b["ids"].to(dev), L, tl, blank=0, reduction="sum", zero_infinity=True) / tl.sum()
    return loss, {}

def s2_loaders(D, TB, seed):
    return dict(asr=iter(make_loader(ASRText(D["CA"]["tr"]), asr_text_collate, TB["asr"], seed)) if TB["asr"] else iter(()),
                at=iter(make_loader(ATData("train", D["fsd_tr"], D["fsd_tr_y"]), at_collate, TB["at"], seed)) if TB["at"] else iter(()),
                sv=iter(MC.sv_loader(D["TR"], None, TB["sv"], MC.CFG["crop_seconds"], E["num_workers"], seed)) if TB["sv"] else iter(()))

def asr_wer(student, H, df, dev, amp):
    import torch
    hyps = {}; bt = MA.make_batches(df.n_samples.values, 0, 0, False, E["max_batch_seconds"], E["max_batch_utts"])
    with torch.no_grad():
        for b in make_loader(ASRText(df), asr_text_collate, bt, 0):
            fb, fl = b["fb"].to(dev), b["fb_lens"].to(dev)
            with torch.autocast("cuda", dtype=amp): enc, el = student(fb, fl); lg = H["ctc"](enc.repeat_interleave(2, dim=1))
            best = lg.float().argmax(-1).cpu(); L = (el * 2).cpu()
            for k, i in enumerate(b["i"].tolist()): hyps[i] = MA.greedy_decode(best[k, :int(L[k])].tolist())
    return MA.error_rates(list(df.text), [hyps[i] for i in range(len(df))])

def at_map200(student, H, split, fnames, y, dev, amp):
    import torch
    lens = [np.load(MB.audio_dir(split) / f"{f}.npy", mmap_mode="r").shape[0] for f in fnames]; P = np.zeros((len(fnames), 200), np.float32)
    bt = MA.make_batches(np.asarray(lens), 0, 0, False, E["max_batch_seconds"], E["max_batch_utts"])
    with torch.no_grad():
        for b in make_loader(ATData(split, fnames, y), at_collate, bt, 0):
            fb, fl = b["fb"].to(dev), b["fb_lens"].to(dev)
            with torch.autocast("cuda", dtype=amp): enc, el = student(fb, fl); z = H["at"](masked_mean(enc, el)).float()
            P[b["i"].numpy()] = z.cpu().numpy()
    return MB.mean_ap(y, P)[0]

def s2_validate(student, H, D, dev, amp):
    """dev-clean WER (greedy CTC), FSD50K val mAP (200 classes), SV EER on NON-OFFICIAL held-out-speaker trials."""
    was = student.training; student.eval(); H.eval()
    w = asr_wer(student, H, D["CA"]["dv"], dev, amp); m = at_map200(student, H, "val", D["fsd_va"], D["fsd_va_y"], dev, amp); e = sel_eer(student, H["msv"], D, dev, amp)
    student.train(was); H.train(was)
    return dict(asr_dev_wer=w["wer"], asr_dev_cer=w["cer"], at_val_mAP=m, sv_val_eer=e, score=(w["wer"] + (1 - m) + e) / 3.0)

def train_stage2(D, dev, amp, drive, s1_best):
    import torch
    student, H1 = build_joint_model(dev)
    ck1 = torch.load(s1_best, map_location="cpu", weights_only=False); student.load_state_dict(ck1["student"]); H1.load_state_dict(ck1["heads"])
    H = build_stage2_heads(H1, D, dev); del ck1
    enc_named = [("student." + n, p) for n, p in student.named_parameters()]; head_named = [("heads." + n, p) for n, p in H.named_parameters()]
    opt, sch = make_scaled_adam([(enc_named, E["base_lr"] * E["s2_encoder_lr_scale"]), (head_named, E["base_lr"])])
    hopt = torch.optim.Adam([p for _, p in head_named], lr=E["s2_head_lr"])            # head warm-up phase (encoder frozen)
    scaler = torch.amp.GradScaler("cuda", enabled=amp == torch.float16)
    ck = load_latest("stage2", drive)
    if ck is None:
        log("  Stage 2 starts from the BEST Stage-1 checkpoint; new heads; encoder frozen for the first "
            f"{E['s2_head_warmup_updates']} updates, then everything is trained (encoder at {E['s2_encoder_lr_scale']} x base lr)")
        S = dict(epoch=0, k=0, update=0, best=None, bad=0, hist=[], finished=False, lam=dict(asr=1.0, at=1.0, sv=1.0))
        TB = task_batches(D, 0, E["seed"] + 998); L = s2_loaders(D, {t: TB[t][:E["init_loss_batches"]] for t in TB}, 0)
        with torch.no_grad():
            for t in ("asr", "at", "sv"): S["lam"][t] = 1.0 / max(np.mean([float(s2_batch_loss(t, b, student, H, D, dev, amp, True)[0]) for b in L[t]]), 1e-6)
        log(f"  Stage 2 loss scales lambda = {S['lam']}")
    else:
        student.load_state_dict(ck["student"]); H.load_state_dict(ck["heads"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"])
        hopt.load_state_dict(ck["hopt"]); scaler.load_state_dict(ck["scaler"]); S = ck["state"]; set_rng(ck["rng"])
        log(f"  RESUMED Stage 2 at epoch {S['epoch'] + 1}, schedule position {S['k']}, update {S['update']}" + (" (already finished)" if S["finished"] else ""))
        if S["finished"] and S["epoch"] < SC["s2_epochs"]:            # you raised the epoch limit after the stage had finished
            if S["bad"] >= E["s2_patience"] and not E.get("force_continue"):
                log(f"  Stage 2 had stopped EARLY (no validation improvement for {S['bad']} epochs): not continued. "
                    "Set E['force_continue'] = True at the top of Part 2 to continue anyway.")
            else:
                S["finished"] = False; S["bad"] = 0
                log(f"  Stage 2 CONTINUES from epoch {S['epoch']} up to the new limit of {SC['s2_epochs']} epochs "
                    "(same model, optimiser and learning-rate schedule; best.pt is replaced only if a new epoch is better)")
    def pack(): return dict(e2e_version=E2E_VERSION, data_sig=DATA_SIG, stage="stage2", student=student.state_dict(), heads=H.state_dict(), opt=opt.state_dict(),
                            sched=sch.state_dict(), hopt=hopt.state_dict(), scaler=scaler.state_dict(), state=S, rng=rng_state(), zip_args=dict(MC.ZIP_ARGS),
                            n_spk=D["n_spk"], vocab=VOCAB)
    t_start = time.time(); last_ck = time.time()
    while not S["finished"] and S["epoch"] < SC["s2_epochs"]:
        ep = S["epoch"]; TB = task_batches(D, ep, E["seed"] + 7); order = schedule(TB, ep, E["seed"] + 7); L = s2_loaders(D, remaining(TB, order, S["k"]), E["seed"] + 7 + ep)
        sch.step_epoch(ep); student.train(); H.train(); t0 = time.time(); acc = {t: [0.0, 0] for t in L}
        log(f"\n  ---- Stage 2 epoch {ep + 1}/{SC['s2_epochs']}: {len(order)} batches, starting at {S['k']}")
        while S["k"] < len(order):
            task = order[S["k"]]; b = next(L[task]); frozen = S["update"] < E["s2_head_warmup_updates"]
            if frozen: student.eval()                                   # frozen encoder: no dropout / layer skipping, no updates
            else:
                student.train()
                if S["update"] == E["s2_head_warmup_updates"]: log("  head warm-up finished: the encoder is now trained too")
            set_bc(student, max(0, S["update"] - E["s2_head_warmup_updates"]))
            loss, _ = s2_batch_loss(task, b, student, H, D, dev, amp, frozen)
            if not torch.isfinite(loss): raise FloatingPointError(f"non-finite {task} loss at update {S['update']}")
            if frozen:
                hopt.zero_grad(set_to_none=True); (loss * S["lam"][task]).backward(); hopt.step()
            else:
                opt.zero_grad(set_to_none=True); scaler.scale(loss * S["lam"][task]).backward(); scaler.step(opt); scaler.update()
                sch.step_batch(S["update"] - E["s2_head_warmup_updates"] + 1)
            S["update"] += 1; S["k"] += 1; acc[task][0] += float(loss); acc[task][1] += 1
            if S["update"] % E["log_every"] == 0:
                el = time.time() - t0
                log(f"  [S2 ep {ep + 1} | {S['k']}/{len(order)} | upd {S['update']}{' | encoder FROZEN' if frozen else ''}] "
                    + " | ".join(f"{t} {a / max(c, 1):.4f}" for t, (a, c) in acc.items()) + f" | {el / 60:.1f} min")
            if time.time() - last_ck > E["ckpt_every_minutes"] * 60: save_ckpt("stage2", pack(), drive); last_ck = time.time()
            if E["max_session_hours"] and time.time() - t_start > E["max_session_hours"] * 3600:
                save_ckpt("stage2", pack(), drive); drive.wait(); log("  session time limit reached: state saved -> run the cell again to continue"); return None
        R = s2_validate(student, H, D, dev, amp); S["epoch"] += 1; S["k"] = 0
        R.update(epoch=S["epoch"], update=S["update"], minutes=round((time.time() - t0) / 60, 1), **{f"train_{t}": a / max(c, 1) for t, (a, c) in acc.items()})
        S["hist"].append(R); improved = S["best"] is None or R["score"] < S["best"] - 1e-4
        log(f"  === Stage 2 epoch {S['epoch']}: dev-clean WER {R['asr_dev_wer'] * 100:.2f}% (CER {R['asr_dev_cer'] * 100:.2f}%) | FSD50K val mAP "
            f"{R['at_val_mAP'] * 100:.2f}% | SV EER {R['sv_val_eer'] * 100:.2f}% (non-official) | score {R['score']:.4f}"
            + (" (BEST)" if improved else f" (no improvement {S['bad'] + 1}/{E['s2_patience']})") + f" | {R['minutes']} min ===")
        if improved: S["best"] = R["score"]; S["bad"] = 0; save_ckpt("stage2", pack(), drive, best=True)
        else: S["bad"] += 1
        if S["bad"] >= E["s2_patience"]: log("  early stopping (no validation improvement)"); S["finished"] = True
        if S["epoch"] >= SC["s2_epochs"]: S["finished"] = True
        save_ckpt("stage2", pack(), drive); last_ck = time.time()
        drive.write_text("history.csv", pd.DataFrame(S["hist"]).to_csv(index=False))
    drive.wait(); return S



def dry_run_s2(D, dev, amp):
    """2 real batches of every task through Stage 2 (frozen-encoder and full fine-tuning paths) before the long run."""
    import torch
    student, H = build_joint_model(dev); TB = task_batches(D, 0, 123)
    H2 = build_stage2_heads(H, D, dev); L2 = s2_loaders(D, {t: TB[t][:2] for t in TB}, 0)
    o2 = torch.optim.Adam(list(student.parameters()) + list(H2.parameters()), lr=1e-5)
    for t in ("asr", "at", "sv"):
        for i_, b in enumerate(L2[t]):
            loss, _ = s2_batch_loss(t, b, student, H2, D, dev, amp, frozen=(i_ == 0))
            assert torch.isfinite(loss), f"stage 2 {t}: non-finite loss"; o2.zero_grad(set_to_none=True); loss.backward(); o2.step()
        log(f"    Stage 2 {t.upper()} batch OK (loss {float(loss):.4f}; frozen-encoder and full fine-tuning paths)")
    del student, H, H2, o2; gc.collect(); torch.cuda.empty_cache()

if not W("dry_run_s2_passed.json").exists() or json.load(open(W("dry_run_s2_passed.json"))).get("data_sig") != DATA_SIG:
    log("  DRY RUN of Stage 2 (2 real batches per task: frozen-encoder and full fine-tuning paths) ...")
    dry_run_s2(D, DEV, AMP); MA.atomic_json(dict(data_sig=DATA_SIG), W("dry_run_s2_passed.json"))

STEP(10, NSTEP, f"STAGE 2 — multi-task fine-tuning with labels: ASR (CTC) + AT (FSD50K, 200 classes) + SV (speaker classification) (max {SC['s2_epochs']} epochs)")
_s1best = W("stage1", "best.pt")
if not _s1best.exists() and (DRV1.dir / "best.pt").exists(): shutil.copyfile(DRV1.dir / "best.pt", _s1best)
S2 = train_stage2(D, DEV, AMP, DRV2, _s1best)
if S2 is None or not S2["finished"]: raise SystemExit("Stage 2 not finished in this session: run the cell again to continue")
DONE(10, f"STAGE 2 finished after {S2['epoch']} epochs")

STEP(11, NSTEP, "FINAL EVALUATION of the best Stage-2 model on the OFFICIAL test sets: LibriSpeech test-clean WER, FSD50K eval mAP, VoxCeleb1-O EER")
_ck = torch.load(W("stage2", "best.pt") if W("stage2", "best.pt").exists() else DRV2.dir / "best.pt", map_location="cpu", weights_only=False)
_st, _h1 = build_joint_model(DEV); _st.load_state_dict(_ck["student"]); _hf = build_stage2_heads(_h1, D, DEV); _hf.load_state_dict(_ck["heads"]); _st.eval(); _hf.eval()
FINAL = dict(best_stage2_epoch=_ck["state"]["hist"][-1]["epoch"] if _ck["state"]["hist"] else None)
FINAL["asr_dev_clean"] = asr_wer(_st, _hf, LS_DV, DEV, AMP)
try:
    _tc = MA.inventory("test-clean").reset_index(drop=True)                                 # official test-clean (used ONLY here, once)
    MA.materialize(_tc, "test-clean"); FINAL["asr_test_clean"] = asr_wer(_st, _hf, _tc, DEV, AMP)
except Exception as e: log(f"  (test-clean WER not computed: {type(e).__name__}: {str(e)[:160]})")
FINAL["at_val_mAP_200"] = at_map200(_st, _hf, "val", list(FSD_VA), FSD_VA_Y, DEV, AMP)
if E["fsd_eval_split"] and "eval" in FSD["splits"]:
    _ei = list(range(len(FSD["splits"]["eval"]["fnames"]))); MB.ensure_audio(FSD, "eval", _ei)
    FINAL["at_eval_mAP_200"] = at_map200(_st, _hf, "eval", FSD["splits"]["eval"]["fnames"], FSD["splits"]["eval"]["y"], DEV, AMP)
FINAL["sv_selection_eer"] = sel_eer(_st, _hf["msv"], D, DEV, AMP)
if _have_trials:
    _Ete = MC.student_embeddings(_st, _hf["msv"], TE, None, DEV, AMP); FINAL["sv_official"] = MC.official_eer(_Ete, TE, "MT2KD STUDENT (after Stage 2)")
FINAL["teacher_sv_official"] = TEACHER_SV; FINAL["teacher_at_val_mAP_mapped"] = _tm
DONE(11, "final evaluation finished")

STEP(12, NSTEP, "SUMMARY + reports on Drive")
_o = (FINAL.get("sv_official") or {}); _o = next(iter(_o.values()), {}) if _o else {}
log(f"  ASR  : dev-clean WER {FINAL['asr_dev_clean']['wer'] * 100:.2f}% (CER {FINAL['asr_dev_clean']['cer'] * 100:.2f}%)"
    + (f" | test-clean WER {FINAL['asr_test_clean']['wer'] * 100:.2f}%" if "asr_test_clean" in FINAL else "") + "  [greedy CTC, characters, no language model]")
log(f"  AT   : FSD50K val mAP {FINAL['at_val_mAP_200'] * 100:.2f}%" + (f" | eval mAP {FINAL['at_eval_mAP_200'] * 100:.2f}%" if "at_eval_mAP_200" in FINAL else "") + "  [200 classes]")
log(f"  SV   : VoxCeleb1-O EER {(_o.get('eer') or float('nan')) * 100:.3f}% ({_o.get('status', 'n/a')}) | non-official selection EER {FINAL['sv_selection_eer'] * 100:.2f}%")
log(f"  scale '{SCALE}': pilot-size data -> numbers are far from the paper's (which used LibriSpeech 960 h, AudioSet, VoxCeleb1+2 and a transducer);"
    " they are the reference for comparing teachers / enhancements under identical settings.")
DRV2.write_json("final_report.json", dict(stage="final", e2e_version=E2E_VERSION, scale=SCALE, data_sig=DATA_SIG, final=FINAL, stage2_history=S2["hist"],
                                         deviations_from_paper=["teachers: WavLM-Large / SSLAM / ReDimNet2-B6 (enhancement) instead of Whisper / BEATs / ECAPA",
                                                                "Stage 2 ASR: CTC with characters + greedy decoding instead of a neural transducer",
                                                                "Stage 2 AT on FSD50K (200 classes) instead of AudioSet; SV on a VoxCeleb1 subset",
                                                                "batches contain one task each (interleaved); every dataset is used completely in every epoch",
                                                                "fixed loss balancing by initial loss values; no checkpoint averaging"]))
DRV2.wait(); DRV1.wait()
DONE(12, f"ALL DONE. Reports: {E['drive_out']}/stage1/final_report.json and {E['drive_out']}/stage2/final_report.json")
